# 1. Setup Paths and Source URLs

Define the working data directory and the remote parquet sources used throughout the notebook.

In [1]:
from pathlib import Path

import polars as pl

root = Path.cwd().resolve()
data_dir = root/ "data"
data_dir.mkdir(parents=True, exist_ok=True)

daioe_source: str = (
    "https://raw.githubusercontent.com/joseph-data/AI_Econ_daioe_years/development/"
    "data/daioe_scb_years_all_levels.parquet"
)

scb_source: str = (
        "https://raw.githubusercontent.com/joseph-data/AI_Econ_daioe_months_v2/daioe_pull/"
        "data/scb_months.parquet"

)

def _pct_chg(current: pl.Expr, shifted: pl.Expr) -> pl.Expr:
    return (
        pl.when(shifted.is_not_null() & shifted.ne(0))
        .then((current / shifted - 1) * 100)
        .otherwise(None)
    )

# 2. Load DAIOE and SCB Lazily

Read both datasets as lazy Polars frames so the transformations can stay efficient until collection or export.

In [2]:
daioe_lf = pl.scan_parquet(
    daioe_source,
)

scb_lf = pl.scan_parquet(
    scb_source,
)

# 3. Quick Sanity Checks

Preview both inputs to confirm the expected columns, types, and general structure before transforming them.

In [3]:
print(daioe_lf.head(5).collect())

shape: (5, 65)
┌───────┬───────────┬───────┬───────┬───┬──────────────┬──────────────┬──────────────┬─────────────┐
│ level ┆ ssyk_code ┆ age   ┆ sex   ┆ … ┆ daioe_lngmod ┆ daioe_transl ┆ daioe_speech ┆ daioe_genai │
│ ---   ┆ ---       ┆ ---   ┆ ---   ┆   ┆ _Level_Expos ┆ at_Level_Exp ┆ rec_Level_Ex ┆ _Level_Expo │
│ str   ┆ str       ┆ str   ┆ str   ┆   ┆ ure          ┆ osure        ┆ posure       ┆ sure        │
│       ┆           ┆       ┆       ┆   ┆ ---          ┆ ---          ┆ ---          ┆ ---         │
│       ┆           ┆       ┆       ┆   ┆ i8           ┆ i8           ┆ i8           ┆ i8          │
╞═══════╪═══════════╪═══════╪═══════╪═══╪══════════════╪══════════════╪══════════════╪═════════════╡
│ SSYK3 ┆ 137       ┆ 16-24 ┆ men   ┆ … ┆ 3            ┆ 3            ┆ 3            ┆ 3           │
│ SSYK3 ┆ 344       ┆ 60-64 ┆ women ┆ … ┆ 4            ┆ 5            ┆ 5            ┆ 4           │
│ SSYK3 ┆ 421       ┆ 25-29 ┆ men   ┆ … ┆ 4            ┆ 5            ┆ 5   

In [4]:
print(scb_lf.head(5).collect())

shape: (5, 5)
┌────────┬─────┬──────────┬───────┬────────────┐
│ code_1 ┆ sex ┆ month    ┆ value ┆ occupation │
│ ---    ┆ --- ┆ ---      ┆ ---   ┆ ---        │
│ str    ┆ str ┆ str      ┆ str   ┆ str        │
╞════════╪═════╪══════════╪═══════╪════════════╡
│ 1      ┆ men ┆ 2015-Jan ┆ 135.1 ┆ Managers   │
│ 1      ┆ men ┆ 2015-Feb ┆ 125.6 ┆ Managers   │
│ 1      ┆ men ┆ 2015-Mar ┆ 120.2 ┆ Managers   │
│ 1      ┆ men ┆ 2015-Apr ┆ 141.6 ┆ Managers   │
│ 1      ┆ men ┆ 2015-May ┆ 137.1 ┆ Managers   │
└────────┴─────┴──────────┴───────┴────────────┘


In [5]:
scb_lf_clean = scb_lf\
    .filter(pl.col("code_1").str.starts_with(0).not_())\
        .with_columns(
    pl.col("month")
      .str.extract(r"^(\d{4})", 1)
      .cast(pl.Int64)
      .alias("year"),
)


print(scb_lf_clean.limit(10).collect())

shape: (10, 6)
┌────────┬─────┬──────────┬───────┬────────────┬──────┐
│ code_1 ┆ sex ┆ month    ┆ value ┆ occupation ┆ year │
│ ---    ┆ --- ┆ ---      ┆ ---   ┆ ---        ┆ ---  │
│ str    ┆ str ┆ str      ┆ str   ┆ str        ┆ i64  │
╞════════╪═════╪══════════╪═══════╪════════════╪══════╡
│ 1      ┆ men ┆ 2015-Jan ┆ 135.1 ┆ Managers   ┆ 2015 │
│ 1      ┆ men ┆ 2015-Feb ┆ 125.6 ┆ Managers   ┆ 2015 │
│ 1      ┆ men ┆ 2015-Mar ┆ 120.2 ┆ Managers   ┆ 2015 │
│ 1      ┆ men ┆ 2015-Apr ┆ 141.6 ┆ Managers   ┆ 2015 │
│ 1      ┆ men ┆ 2015-May ┆ 137.1 ┆ Managers   ┆ 2015 │
│ 1      ┆ men ┆ 2015-Jun ┆ 112.8 ┆ Managers   ┆ 2015 │
│ 1      ┆ men ┆ 2015-Jul ┆ 142.5 ┆ Managers   ┆ 2015 │
│ 1      ┆ men ┆ 2015-Aug ┆ 137.1 ┆ Managers   ┆ 2015 │
│ 1      ┆ men ┆ 2015-Sep ┆ 120.7 ┆ Managers   ┆ 2015 │
│ 1      ┆ men ┆ 2015-Oct ┆ 150.0 ┆ Managers   ┆ 2015 │
└────────┴─────┴──────────┴───────┴────────────┴──────┘


# 4. Compute 1-month, 3-months and 6-months Change

Clean the SCB monthly employment counts and create short-run absolute and percentage change measures by occupation and sex.

In [6]:

change_keys = [col for col in scb_lf_clean.collect_schema().names() if col != "value"]
time_col = "month"
group_keys = [col for col in change_keys if col != time_col]

raw_emp = pl.col("value")
emp = pl.col("emp_count")

scb_lazy_lf_changes = (
    scb_lf_clean
    .with_columns(pl.col("value").cast(pl.Float64, strict=False))
    .group_by(change_keys)
    .agg(raw_emp.sum().alias("emp_count"))
    .with_columns(
        pl.col(time_col).str.strptime(pl.Date, "%Y-%b", strict=False).alias("_month_date"),
    )
    # Pre-compute shifted values
    .with_columns(
        emp.shift(1).over(group_keys, order_by="_month_date").alias("_emp_1m"),
        emp.shift(3).over(group_keys, order_by="_month_date").alias("_emp_3m"),
        emp.shift(6).over(group_keys, order_by="_month_date").alias("_emp_6m"),
    )
    # Absolute and percentage changes
    .with_columns(
        (emp - pl.col("_emp_1m")).alias("chg_1m"),
        (emp - pl.col("_emp_3m")).alias("chg_3m"),
        (emp - pl.col("_emp_6m")).alias("chg_6m"),
        _pct_chg(emp, pl.col("_emp_1m")).alias("pct_chg_1m"),
        _pct_chg(emp, pl.col("_emp_3m")).alias("pct_chg_3m"),
        _pct_chg(emp, pl.col("_emp_6m")).alias("pct_chg_6m"),
    )
    .sort(
        by=[
            "code_1",
            "sex",
            "occupation",
            "_month_date",
        ],
    )
    .drop("_month_date", "_emp_1m", "_emp_3m", "_emp_6m")
)

scb_lazy_lf_changes.collect_schema()

Schema([('code_1', String),
        ('sex', String),
        ('month', String),
        ('occupation', String),
        ('year', Int64),
        ('emp_count', Float64),
        ('chg_1m', Float64),
        ('chg_3m', Float64),
        ('chg_6m', Float64),
        ('pct_chg_1m', Float64),
        ('pct_chg_3m', Float64),
        ('pct_chg_6m', Float64)])

In [7]:
scb_lazy_lf_changes.head(10).collect()

code_1,sex,month,occupation,year,emp_count,chg_1m,chg_3m,chg_6m,pct_chg_1m,pct_chg_3m,pct_chg_6m
str,str,str,str,i64,f64,f64,f64,f64,f64,f64,f64
"""1""","""men""","""2015-Jan""","""Managers""",2015,438.0,null,null,null,null,null,null
"""1""","""men""","""2015-Feb""","""Managers""",2015,412.7,-25.3,null,null,-5.776256,null,null
"""1""","""men""","""2015-Mar""","""Managers""",2015,393.7,-19.0,null,null,-4.603828,null,null
"""1""","""men""","""2015-Apr""","""Managers""",2015,452.1,58.4,14.1,null,14.83363,3.219178,null
"""1""","""men""","""2015-May""","""Managers""",2015,447.5,-4.6,34.8,null,-1.017474,8.432275,null
"""1""","""men""","""2015-Jun""","""Managers""",2015,372.2,-75.3,-21.5,null,-16.826816,-5.461011,null
"""1""","""men""","""2015-Jul""","""Managers""",2015,455.4,83.2,3.3,17.4,22.353573,0.729927,3.972603
"""1""","""men""","""2015-Aug""","""Managers""",2015,443.8,-11.6,-3.7,31.1,-2.547211,-0.826816,7.53574
"""1""","""men""","""2015-Sep""","""Managers""",2015,395.1,-48.7,22.9,1.4,-10.973411,6.152606,0.355601


# 5. Prepare DAIOE Level-1 Aggregates

Filter the DAIOE data to SSYK1 and aggregate yearly exposure metrics so they can be joined onto the SCB panel.

In [8]:
weighted_daioe = daioe_lf\
    .filter(
        pl.col("level") == "SSYK1",
        )\
        .select(
            pl.col(["level", "ssyk_code", "year", "weight_sum"]),
            pl.col("^daioe_.*$"),
            pl.col("^pctl_daioe_.*$"),
            )\
            .group_by(["level", "ssyk_code", "year"])\
                .agg([
                    pl.col("weight_sum").mean().cast(pl.Int64),
                    pl.col("^daioe_.*$").mean(),
                    pl.col("^pctl_daioe_.*$").mean(),
                    ])

weighted_daioe.limit(10).collect()

level,ssyk_code,year,weight_sum,daioe_allapps_avg,daioe_stratgames_avg,daioe_videogames_avg,daioe_imgrec_avg,daioe_imgcompr_avg,daioe_imggen_avg,daioe_readcompr_avg,daioe_lngmod_avg,daioe_translat_avg,daioe_speechrec_avg,daioe_genai_avg,daioe_allapps_wavg,daioe_stratgames_wavg,daioe_videogames_wavg,daioe_imgrec_wavg,daioe_imgcompr_wavg,daioe_imggen_wavg,daioe_readcompr_wavg,daioe_lngmod_wavg,daioe_translat_wavg,daioe_speechrec_wavg,daioe_genai_wavg,daioe_allapps_Level_Exposure,daioe_stratgames_Level_Exposure,daioe_videogames_Level_Exposure,daioe_imgrec_Level_Exposure,daioe_imgcompr_Level_Exposure,daioe_imggen_Level_Exposure,daioe_readcompr_Level_Exposure,daioe_lngmod_Level_Exposure,daioe_translat_Level_Exposure,daioe_speechrec_Level_Exposure,daioe_genai_Level_Exposure,pctl_daioe_allapps_avg,pctl_daioe_stratgames_avg,pctl_daioe_videogames_avg,pctl_daioe_imgrec_avg,pctl_daioe_imgcompr_avg,pctl_daioe_imggen_avg,pctl_daioe_readcompr_avg,pctl_daioe_lngmod_avg,pctl_daioe_translat_avg,pctl_daioe_speechrec_avg,pctl_daioe_genai_avg,pctl_daioe_allapps_wavg,pctl_daioe_stratgames_wavg,pctl_daioe_videogames_wavg,pctl_daioe_imgrec_wavg,pctl_daioe_imgcompr_wavg,pctl_daioe_imggen_wavg,pctl_daioe_readcompr_wavg,pctl_daioe_lngmod_wavg,pctl_daioe_translat_wavg,pctl_daioe_speechrec_wavg,pctl_daioe_genai_wavg
str,str,i64,i64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
"""SSYK1""","""3""",2014,548032,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,5.0,5.0,5.0,5.0,3.0,3.0,5.0,5.0,3.0,5.0,5.0,87.5,87.5,87.5,87.5,50.0,50.0,87.5,87.5,50.0,87.5,87.5,87.5,87.5,87.5,87.5,50.0,50.0,87.5,87.5,50.0,87.5,87.5
"""SSYK1""","""5""",2015,939548,8.299752,0.181875,2.84279,0.151194,NaN,0.019695,0.003724,0.01667,0.00248,0.19421,0.060471,8.106642,0.179994,2.774004,0.147831,NaN,0.019486,0.003666,0.016365,0.002383,0.185621,0.059662,1.0,1.0,1.0,1.0,3.0,2.0,2.0,2.0,2.0,2.0,2.0,0.0,12.5,12.5,12.5,50.0,37.5,37.5,37.5,37.5,37.5,37.5,0.0,12.5,12.5,0.0,50.0,25.0,37.5,37.5,37.5,37.5,37.5
"""SSYK1""","""6""",2022,36769,24.661914,0.232034,5.203614,0.374991,0.132937,0.511006,0.135699,0.424794,0.022764,0.282106,1.589928,24.940471,0.232277,5.282947,0.375138,0.13407,0.515241,0.138125,0.430478,0.023336,0.28685,1.607191,1.0,2.0,2.0,2.0,1.0,1.0,2.0,2.0,2.0,2.0,2.0,12.5,25.0,37.5,25.0,12.5,12.5,25.0,12.5,12.5,12.5,25.0,12.5,25.0,37.5,25.0,12.5,12.5,25.0,25.0,25.0,25.0,25.0
"""SSYK1""","""7""",2023,445323,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,5.0,5.0,5.0,5.0,5.0,5.0,5.0,5.0,5.0,5.0,5.0,87.5,87.5,87.5,87.5,87.5,87.5,87.5,87.5,87.5,87.5,87.5,87.5,87.5,87.5,87.5,87.5,87.5,87.5,87.5,87.5,87.5,87.5
"""SSYK1""","""4""",2015,347608,10.090038,0.23664,3.007695,0.203123,NaN,0.026565,0.005914,0.026428,0.00395,0.293885,0.088349,10.428546,0.248565,3.15925,0.211541,NaN,0.02729,0.006048,0.026745,0.00393,0.286352,0.090084,4.0,4.0,2.0,4.0,3.0,3.0,3.0,4.0,4.0,4.0,3.0,62.5,50.0,25.0,62.5,50.0,50.0,50.0,62.5,62.5,62.5,50.0,62.5,62.5,25.0,62.5,50.0,50.0,50.0,62.5,62.5,62.5,50.0
"""SSYK1""","""7""",2018,402690,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,5.0,5.0,5.0,5.0,5.0,5.0,5.0,5.0,5.0,5.0,5.0,87.5,87.5,87.5,87.5,87.5,87.5,87.5,87.5,87.5,87.5,87.5,87.5,87.5,87.5,87.5,87.5,87.5,87.5,87.5,87.5,87.5,87.5
"""SSYK1""","""3""",2024,670251,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,5.0,5.0,5.0,5.0,5.0,5.0,5.0,5.0,5.0,5.0,5.0,87.5,87.5,87.5,87.5,87.5,87.5,87.5,87.5,87.5,87.5,87.5,87.5,87.5,87.5,87.5,87.5,87.5,87.5,87.5,87.5,87.5,87.5
"""SSYK1""","""3""",2015,553719,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,5.0,5.0,5.0,5.0,3.0,5.0,5.0,5.0,5.0,5.0,5.0,87.5,87.5,87.5,87.5,50.0,87.5,87.5,87.5,87.5,87.5,87.5,87.5,87.5,87.5,87.

# 6. Align DAIOE Years to SCB Coverage

Extend the yearly DAIOE table forward to match the latest year present in the SCB data.

In [9]:
## Here I extend the years to Latest according to the pulled SCB data (2024, yearly)

base = weighted_daioe

# Extend years to latest available in SCB data (2024, yearly)
DAIOE_MAX_YEAR = base.select(pl.max("year")).collect().item()

SCB_MAX_YEAR = scb_lazy_lf_changes.select(pl.max("year")).collect().item()

missing_years = list(range(DAIOE_MAX_YEAR + 1, SCB_MAX_YEAR + 1))

daioe_lazy_lf_extended = (
    base
    if not missing_years
    else pl.concat(
        [
            base,
            base
            .filter(pl.col("year") == DAIOE_MAX_YEAR)
            .drop("year")
            .join(pl.LazyFrame({"year": missing_years}), how="cross")
            .select(base.collect_schema().names()),
        ],
        how="vertical",
    )
)

In [10]:
daioe_lazy_lf_extended.filter(pl.col("year") == SCB_MAX_YEAR).limit(10).collect()

level,ssyk_code,year,weight_sum,daioe_allapps_avg,daioe_stratgames_avg,daioe_videogames_avg,daioe_imgrec_avg,daioe_imgcompr_avg,daioe_imggen_avg,daioe_readcompr_avg,daioe_lngmod_avg,daioe_translat_avg,daioe_speechrec_avg,daioe_genai_avg,daioe_allapps_wavg,daioe_stratgames_wavg,daioe_videogames_wavg,daioe_imgrec_wavg,daioe_imgcompr_wavg,daioe_imggen_wavg,daioe_readcompr_wavg,daioe_lngmod_wavg,daioe_translat_wavg,daioe_speechrec_wavg,daioe_genai_wavg,daioe_allapps_Level_Exposure,daioe_stratgames_Level_Exposure,daioe_videogames_Level_Exposure,daioe_imgrec_Level_Exposure,daioe_imgcompr_Level_Exposure,daioe_imggen_Level_Exposure,daioe_readcompr_Level_Exposure,daioe_lngmod_Level_Exposure,daioe_translat_Level_Exposure,daioe_speechrec_Level_Exposure,daioe_genai_Level_Exposure,pctl_daioe_allapps_avg,pctl_daioe_stratgames_avg,pctl_daioe_videogames_avg,pctl_daioe_imgrec_avg,pctl_daioe_imgcompr_avg,pctl_daioe_imggen_avg,pctl_daioe_readcompr_avg,pctl_daioe_lngmod_avg,pctl_daioe_translat_avg,pctl_daioe_speechrec_avg,pctl_daioe_genai_avg,pctl_daioe_allapps_wavg,pctl_daioe_stratgames_wavg,pctl_daioe_videogames_wavg,pctl_daioe_imgrec_wavg,pctl_daioe_imgcompr_wavg,pctl_daioe_imggen_wavg,pctl_daioe_readcompr_wavg,pctl_daioe_lngmod_wavg,pctl_daioe_translat_wavg,pctl_daioe_speechrec_wavg,pctl_daioe_genai_wavg
str,str,i64,i64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
"""SSYK1""","""8""",2026,347858,27.739882,0.247078,5.853991,0.390965,0.181807,0.519702,0.205744,0.502361,0.020999,0.45489,1.683537,27.923731,0.242952,5.752996,0.406446,0.189149,0.538344,0.209542,0.51159,0.021722,0.46303,1.728679,2.0,2.0,4.0,2.0,2.0,2.0,1.0,1.0,1.0,1.0,1.0,25.0,37.5,62.5,37.5,25.0,25.0,0.0,0.0,0.0,0.0,0.0,25.0,37.5,62.5,37.5,37.5,25.0,12.5,12.5,12.5,12.5,12.5
"""SSYK1""","""2""",2026,1320814,36.43596,0.300564,4.399623,0.47834,0.260963,0.816614,0.487964,1.17935,0.046742,0.838765,3.267108,37.544198,0.310943,4.406035,0.488998,0.268232,0.848918,0.509024,1.23735,0.048721,0.877804,3.415668,3.0,4.0,1.0,3.0,3.0,4.0,4.0,4.0,3.0,3.0,4.0,50.0,62.5,0.0,50.0,50.0,62.5,62.5,50.0,50.0,50.0,62.5,50.0,62.5,12.5,50.0,50.0,62.5,62.5,62.5,50.0,50.0,62.5
"""SSYK1""","""9""",2026,302501,26.59692,0.22139,5.230601,0.35906,0.174449,0.483004,0.219191,0.56153,0.023751,0.501279,1.715393,25.796495,0.217758,5.38268,0.351348,0.167012,0.449842,0.19825,0.508761,0.020827,0.451128,1.577562,1.0,1.0,3.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,0.0,0.0,50.0,0.0,0.0,0.0,12.5,25.0,25.0,25.0,12.5,0.0,0.0,50.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
"""SSYK1""","""6""",2026,49511,26.884467,0.232034,5.207594,0.377393,0.181194,0.51326,0.222071,0.55075,0.023019,0.484068,1.751659,27.164666,0.232404,5.269809,0.376954,0.182427,0.516475,0.22626,0.559326,0.023605,0.492969,1.771117,1.0,2.0,2.0,2.0,1.0,1.0,2.0,2.0,2.0,2.0,2.0,12.5,25.0,37.5,25.0,12.5,12.5,25.0,12.5,12.5,12.5,25.0,12.5,25.0,37.5,25.0,12.5,12.5,25.0,25.0,25.0,25.0,25.0
"""SSYK1""","""1""",2026,356245,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,5.0,5.0,5.0,5.0,5.0,5.0,5.0,5.0,5.0,5.0,5.0,87.5,87.5,87.5,87.5,87.5,87.5,87.5,87.5,87.5,87.5,87.5,87.5,87.5,87.5,87.5,87.5,87.5,87.5,87.5,87.5,87.5,87.5
"""SSYK1""","""7""",2026,438278,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,5.0,5.0,5.0,5.0,5.0,5.0,5.0,5.0,5.0,5.0,5.0,87.5,87.5,87.5,87.5,87.5,87.5,87.5,87.5,87.5,87.5,87.5,87.5,87.5,87.5,87.5,87.5,87.5,87.5,87.5,87.5,87.5,87.5
"""SSYK1""","""4""",2026,392453,37.751187,0.296127,4.749615,0.483484,0.263563,0.73693,0.467952,1.200227,0.051407,0.996461,3.156405,38.409884,0.309081,4.966474,0.501575,0.270525,0.755581,0.473785,1.200682,0.050422,0.955652,3.191323,4.0,3.0,2.0,4.0,4.0,3.0,3.0,3.0,4.0,4.0,3.0,62.5,50.0,25.0,62.5,62.5,50.0,50.0,62.5,62.5,62.5,50.0,62.5,50.0,25.0,62.5,62.5,50.0,50.0,50.0,62.5,62.5,50.0
"""SSYK1""",""

# 7. Merge SCB Monthly Data with the DAIOE Weights

Join the monthly SCB employment panel with the yearly DAIOE features using the shared occupation code and year.

In [11]:
scb_months_lf =scb_lazy_lf_changes\
                .join(
                    daioe_lazy_lf_extended,
                    left_on=["code_1", "year"],
                    right_on=["ssyk_code", "year"],
                    how = "left",
                )\
                    .drop("level")

scb_months_lf.limit(10).collect()

code_1,sex,month,occupation,year,emp_count,chg_1m,chg_3m,chg_6m,pct_chg_1m,pct_chg_3m,pct_chg_6m,weight_sum,daioe_allapps_avg,daioe_stratgames_avg,daioe_videogames_avg,daioe_imgrec_avg,daioe_imgcompr_avg,daioe_imggen_avg,daioe_readcompr_avg,daioe_lngmod_avg,daioe_translat_avg,daioe_speechrec_avg,daioe_genai_avg,daioe_allapps_wavg,daioe_stratgames_wavg,daioe_videogames_wavg,daioe_imgrec_wavg,daioe_imgcompr_wavg,daioe_imggen_wavg,daioe_readcompr_wavg,daioe_lngmod_wavg,daioe_translat_wavg,daioe_speechrec_wavg,daioe_genai_wavg,daioe_allapps_Level_Exposure,daioe_stratgames_Level_Exposure,daioe_videogames_Level_Exposure,daioe_imgrec_Level_Exposure,daioe_imgcompr_Level_Exposure,daioe_imggen_Level_Exposure,daioe_readcompr_Level_Exposure,daioe_lngmod_Level_Exposure,daioe_translat_Level_Exposure,daioe_speechrec_Level_Exposure,daioe_genai_Level_Exposure,pctl_daioe_allapps_avg,pctl_daioe_stratgames_avg,pctl_daioe_videogames_avg,pctl_daioe_imgrec_avg,pctl_daioe_imgcompr_avg,pctl_daioe_imggen_avg,pctl_daioe_readcompr_avg,pctl_daioe_lngmod_avg,pctl_daioe_translat_avg,pctl_daioe_speechrec_avg,pctl_daioe_genai_avg,pctl_daioe_allapps_wavg,pctl_daioe_stratgames_wavg,pctl_daioe_videogames_wavg,pctl_daioe_imgrec_wavg,pctl_daioe_imgcompr_wavg,pctl_daioe_imggen_wavg,pctl_daioe_readcompr_wavg,pctl_daioe_lngmod_wavg,pctl_daioe_translat_wavg,pctl_daioe_speechrec_wavg,pctl_daioe_genai_wavg
str,str,str,str,i64,f64,f64,f64,f64,f64,f64,f64,i64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
"""2""","""men""","""2024-Aug""","""Occupations requiring advanced…",2024,2035.0,88.4,-7.2,49.5,4.541251,-0.352561,2.493075,1320814,36.43596,0.300564,4.399623,0.47834,0.260963,0.816614,0.487964,1.17935,0.046742,0.838765,3.267108,37.544198,0.310943,4.406035,0.488998,0.268232,0.848918,0.509024,1.23735,0.048721,0.877804,3.415668,3.0,4.0,1.0,3.0,3.0,4.0,4.0,4.0,3.0,3.0,4.0,50.0,62.5,0.0,50.0,50.0,62.5,62.5,50.0,50.0,50.0,62.5,50.0,62.5,12.5,50.0,50.0,62.5,62.5,62.5,50.0,50.0,62.5
"""6""","""women""","""2021-Sep""","""Agricultural, horticultural, f…",2021,27.1,-31.2,1.2,27.1,-53.516295,4.633205,null,35749,23.047838,0.232034,5.18845,0.362916,0.116573,0.488727,0.114432,0.216746,0.022326,0.262345,1.224159,23.296192,0.232092,5.27097,0.362973,0.117488,0.492305,0.116185,0.219066,0.022842,0.266355,1.234956,2.0,2.0,2.0,2.0,1.0,1.0,2.0,2.0,2.0,2.0,2.0,12.5,25.0,37.5,25.0,12.5,12.5,25.0,12.5,12.5,12.5,25.0,25.0,25.0,37.5,25.0,12.5,12.5,25.0,25.0,25.0,25.0,25.0
"""4""","""women""","""2023-Apr""","""Administration and customer se…",2023,725.0,96.9,2.6,null,15.42748,0.359911,null,401274,37.751187,0.296127,4.749615,0.483484,0.263563,0.73693,0.467952,1.200227,0.051407,0.996461,3.156405,38.433513,0.309359,4.970153,0.501849,0.270662,0.755757,0.474172,1.201566,0.050452,0.95607,3.192977,4.0,3.0,2.0,4.0,4.0,3.0,3.0,3.0,4.0,4.0,3.0,62.5,50.0,25.0,62.5,62.5,50.0,50.0,62.5,62.5,62.5,50.0,62.5,50.0,25.0,62.5,62.5,50.0,50.0,50.0,62.5,62.5,50.0
"""7""","""women""","""2025-Jun""","""Building and manufacturing wor…",2025,27.8,-44.6,0.8,null,-61.60221,2.962963,null,438278,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,5.0,5.0,5.0,5.0,5.0,5.0,5.0,5.0,5.0,5.0,5.0,87.5,87.5,87.5,87.5,87.5,87.5,87.5,87.5,87.5,87.5,87.5,87.5,87.5,87.5,87.5,87.5,87.5,87.5,87.5,87.5,87.5,87.5
"""1""","""men""","""2022-Mar""","""Managers""",2022,547.6,-54.7,null,null,-9.081853,null,null,319626,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,5.0,5.0,5.0,5.0,5.0,5.0,5.0,5.0,5.0,5.0,5.0,87.5,87.5,87.5,87.5,87.5,87.5,87.5,87.5,87.5,87.5,87.5,87.5,87.5,87.5,87.5,87.5,87.5,87.5,87.5,87.5,87.5,87.5
"""5""","""women""","""2015-Jul""","""Service, care and shop sales w…",2015,1821.0,151.3,145.7,172.7,9.061508,8.69695,10.477462,939548,8.299752,0.181875,2.84279,0.151194,NaN,0.019695,0.003724

# 8. Export the Cleaned Monthly Output

Write the final merged monthly dataset to parquet for downstream analysis.

In [12]:
scb_months_lf.sink_parquet(data_dir / "scb_months_lvl1.parquet")